## 单元刚度矩阵

In [3]:
def stiffness_1d(EA, L):
    '''
    2自由度pin-jointed二节点单元
    全局刚度矩阵
    '''        
    k = EA / L * np.array([[1, -1],
                           [-1, 1]])
    
    return k
    

## 组装整体刚度矩阵

In [4]:
def assemble_K_1d_chain(ke_list):
    """
    组装一维串联 bar element 的总体刚度矩阵

    默认连接关系：
        element 1: node 1 - node 2
        element 2: node 2 - node 3
        element 3: node 3 - node 4
        ...

    Parameters
    ----------
    ke_list : list
        单元刚度矩阵列表，每个 ke 为 2x2 矩阵

    Returns
    -------
    K : ndarray
        总体刚度矩阵
    """

    n_elements = len(ke_list)
    n_nodes = n_elements + 1

    K = np.zeros((n_nodes, n_nodes))

    for e, ke in enumerate(ke_list):
        ke = np.asarray(ke, dtype=float)

        if ke.shape != (2, 2):
            raise ValueError("Each 1D element stiffness matrix must be 2x2.")

        # element e 连接 node e 和 node e+1
        # Python 编号从 0 开始
        dofs = [e, e + 1]

        K[np.ix_(dofs, dofs)] += ke

    return K

## 求解器函数

In [ ]:
def solve_structure_1d(K, force_bc=None, disp_bc=None, numbering="one", check=True):
    """
    求解一维串联 bar structure 的总体刚度方程 K d = F

    Parameters
    ----------
    K : ndarray
        总体刚度矩阵

    force_bc : dict
        已知节点力，直接使用节点编号，例如：
        {
            2: P
        }

    disp_bc : dict
        已知节点位移，直接使用节点编号，例如：
        {
            1: 0.0,
            3: 0.0
        }

    numbering : str
        "one"  表示节点编号从 1 开始
        "zero" 表示节点编号从 0 开始

    Returns
    -------
    d : ndarray
        完整节点位移向量

    reaction : ndarray
        完整节点反力向量

    free_dofs : ndarray
        自由自由度编号

    fixed_dofs : ndarray
        约束自由度编号
    """

    K = np.asarray(K, dtype=float)

    if K.shape[0] != K.shape[1]:
        raise ValueError("K must be a square matrix.")

    n_dof = K.shape[0]

    def node_to_index(node):
        if numbering == "one":
            return node - 1
        elif numbering == "zero":
            return node
        else:
            raise ValueError("numbering must be 'one' or 'zero'.")

    # -------------------------
    # 1. 定义整体载荷向量 F
    # -------------------------
    F = np.zeros(n_dof)

    if force_bc is not None:
        for node, value in force_bc.items():
            index = node_to_index(node)
            F[index] = value

    # -------------------------
    # 2. 定义整体位移向量 d
    # -------------------------
    if disp_bc is None:
        disp_bc = {}

    d = np.zeros(n_dof)

    fixed_dofs = []

    for node, value in disp_bc.items():
        index = node_to_index(node)
        d[index] = value
        fixed_dofs.append(index)

    fixed_dofs = np.array(sorted(fixed_dofs), dtype=int)

    # -------------------------
    # 3. 区分自由自由度和约束自由度
    # -------------------------
    all_dofs = np.arange(n_dof)
    free_dofs = np.setdiff1d(all_dofs, fixed_dofs)

    # -------------------------
    # 4. 矩阵分块
    # -------------------------
    K_ff = K[np.ix_(free_dofs, free_dofs)]
    K_fc = K[np.ix_(free_dofs, fixed_dofs)]

    F_f = F[free_dofs]
    d_c = d[fixed_dofs]

    # -------------------------
    # 5. 检查约束是否足够
    # -------------------------
    if check:
        rank = np.linalg.matrix_rank(K_ff)

        if rank < K_ff.shape[0]:
            raise ValueError(
                "K_ff is singular. The structure may be under-constrained "
                "or contains a mechanism."
            )

    # -------------------------
    # 6. 求解自由位移
    # -------------------------
    rhs = F_f - K_fc @ d_c
    d_f = np.linalg.solve(K_ff, rhs)

    d[free_dofs] = d_f

    # -------------------------
    # 7. 计算反力
    # -------------------------
    reaction = K @ d - F

    return d, reaction, free_dofs, fixed_dofs